# 01 - PADO-Pauli Quickstart

What this notebook answers:

- What is the Pauli-Propagation Surrogate (PPS) and why "compile once, evaluate many"?
- How do I build a circuit with the `Circuit` builder, compile it with observables, and get expectation values?
- How do I get gradients, evaluate parameter batches, and feed data embeddings?
- How do I control the surrogate accuracy/cost trade-off (truncation), add noise, and cross-check against PennyLane?

## Notebook Guide / 노트북 가이드

- **EN** Read top to bottom; every section is a runnable, self-contained step.
- **KO** 위에서 아래로 읽으세요. 모든 섹션은 독립적으로 실행 가능한 단계입니다.
- **EN** Section flow: concept → build & draw & compile → evaluate → gradients → batches/embeddings → truncation → noise → cross-check.
- **KO** 섹션 순서: 개념 → 회로 작성·draw·컴파일 → 평가 → 그래디언트 → 배치/임베딩 → truncation → 노이즈 → 교차검증.
- **EN** Expected outcome: you can compile and differentiate your own circuit.
- **KO** 기대 결과: 자신의 회로를 컴파일하고 미분할 수 있게 됩니다.

In [11]:
import warnings
warnings.filterwarnings("ignore")

import torch

import padopauli
from padopauli import Circuit

use_cuda = torch.cuda.is_available()
nb_preset = "gpu" if use_cuda else "cpu"
print(f"padopauli {padopauli.__version__}")
print(f"torch {torch.__version__} | cuda available: {use_cuda} | preset: {nb_preset}")


padopauli 2.0.0
torch 2.11.0+rocm7.2 | cuda available: True | preset: gpu


## 1) The core idea

PADO-Pauli evaluates quantum expectation values with a **Pauli-Propagation Surrogate (PPS)**:
the circuit and the observables are compiled **once** into a sparse tensor operator program
(Heisenberg picture back-propagation of the observables + exact zero-filtering of terms that
cannot contribute on the |0...0⟩ input state). After that single compile, evaluating the
program for **any** parameter vector is cheap and fully differentiable.

```
circuit + observables  --compile-->  sparse tensor program  --evaluate-->  ⟨O⟩(θ), ∇⟨O⟩(θ)
        (once)                                                (many times, batched, on GPU)
```

A circuit is built with the **`Circuit` builder** — gate methods chain and return `self`:

- `qc.rx(q, param_idx=i)` / `qc.rzz(q0, q1, param_idx=i)` / generic `qc.rot("XXXY", [0, 1, 2, 3], param_idx=i)`
  — exp(-i θ[i]/2 · P). The **same `param_idx` can be reused** across gates (parameter sharing).
- `qc.rx(q, embedding_idx=j)` — the angle comes from a separate **data vector** instead of θ (section 6).
- `qc.h(q)`, `qc.x(q)`, `qc.s(q)`, `qc.cnot(control, target)`, `qc.cz(q0, q1)` — engine-native Cliffords.
- `qc.depolarizing(q, px, py, pz)`, `qc.amplitude_damping(q, gamma)` — noise channels (section 7).
- `qc.draw()` — ASCII diagram at any time; after compile it also shows the observables.

`Circuit` is a facade over the functional API (`PauliRotation` lists +
`compile_program`), which remains available.

In [12]:
n = 6

# A small hardware-efficient circuit: RX layer -> entangling RZZ ladder -> RX layer
qc = Circuit(n)
pidx = 0
for q in range(n):
    qc.rx(q, param_idx=pidx); pidx += 1
for q in range(n - 1):
    qc.rzz(q, q + 1, param_idx=pidx); pidx += 1
qc.cnot(0, 1)
for q in range(n):
    qc.rx(q, param_idx=pidx); pidx += 1

n_params = qc.n_params
print(f"{len(qc.gates)} gates, {n_params} parameters")
qc.draw()

# Observables: tuples (pauli, qubits[, coeff]); a list of tuples is ONE multi-term observable
observables = [
    ("Z", [0]),                                        # <Z_0>
    ("ZZ", [0, n - 1]),                                # end-to-end correlator
    [("ZZ", [q, q + 1], 1.0) for q in range(n - 1)],   # an Ising-chain energy (one PauliSum)
]

18 gates, 17 parameters
q0:─RXθ₀─RZθ₆───────●───RXθ₁₁────────────
          │         │
q1:─RXθ₁──RZ──RZθ₇──X───RXθ₁₂────────────
               │
q2:─RXθ₂───────RZ──RZθ₈─RXθ₁₃────────────
                    │
q3:─RXθ₃────────────RZ───RZθ₉─RXθ₁₄──────
                          │
q4:─RXθ₄──────────────────RZ──RZθ₁₀─RXθ₁₅
                                │
q5:─RXθ₅────────────────────────RZ──RXθ₁₆


## 2) Compile once, evaluate many

`qc.compile(observables=..., preset=...)` compiles the circuit and stores the program on the
circuit (and **also returns it** — a `CompiledProgram`, the same object the functional
API produces, so both styles mix freely). `preset` picks where the program lives and runs:
`"cpu"`, `"gpu"`, or `"hybrid"` (CPU storage + GPU compute). Presets default to `float64`;
this notebook passes `dtype="float64"` explicitly so the choice is visible.

In [13]:
program = qc.compile(observables=observables, preset=nb_preset, dtype="float64")

theta = torch.zeros(n_params, dtype=torch.float64)
print("expvals at theta=0:", qc.expvals(theta).tolist())   # |0..0> -> all Z-diagonal obs = 1

theta = torch.linspace(0.1, 1.2, n_params, dtype=torch.float64)
ev = qc.expvals(theta)
print("expvals:", [f"{v:+.6f}" for v in ev.tolist()])

propagate:   0%|          | 0/18 [00:00<?, ?it/s]

[PPS Info] Propagation complete. Terms generated: 128


zero-filter:   0%|          | 0/18 [00:00<?, ?it/s]

[PPS Info] Terms retained after pruning: 10 (7.812500% of peak)
expvals at theta=0: [1.0, 1.0, 5.0]
expvals: ['+0.645242', '+0.028977', '+1.109788']


## 3) Gradients

Pass a θ with `requires_grad=True` and call `.backward()` — the default `diff_mode="vjp"`
uses a hand-written vector-Jacobian product; `diff_mode="autograd"` differentiates the
adjoint graph with host autograd. **Both give identical gradients.**

In [ ]:
theta = torch.linspace(0.1, 1.2, n_params, dtype=torch.float64, requires_grad=True)

loss = qc.expvals(theta)[2]          # the chain-energy observable
loss.backward()                      # VJP gradient computation
g_manual = theta.grad.clone()

theta.grad = None
loss = qc.expvals(theta, diff_mode="autograd")[2]
loss.backward()
g_auto = theta.grad.clone()

print("max |grad| difference (vjp vs autograd):",
      float((g_manual - g_auto).abs().max()))
print("first 5 gradient entries:", [f"{v:+.6f}" for v in g_manual[:5].tolist()])

max |grad| difference (vjp vs autograd): 5.551115123125783e-17
first 5 gradient entries: ['-0.320686', '-0.432911', '-0.195304', '-0.017616', '+0.257038']


## 4) Parameter batches

`qc.expvals` accepts a `(batch, n_params)` matrix and returns `(batch, n_obs)` — one compiled
program evaluates many parameter sets at once (this is how optimizer populations and
landscape scans are done).

In [15]:
theta_batch = torch.rand(8, n_params, dtype=torch.float64)
ev_batch = qc.expvals(theta_batch)
print("theta batch", tuple(theta_batch.shape), "-> expvals", tuple(ev_batch.shape))

theta batch (8, 17) -> expvals (8, 3)


## 5) Presets and truncation (accuracy vs cost)

The surrogate is exact by default. For larger circuits you trade accuracy for cost with
**truncation**, set at compile time:

- `qc.compile(..., max_weight=w)` — drop Pauli terms of weight > w during propagation.
- `qc.compile(..., build_thetas=θ0, build_min_abs=ε)` — additionally drop terms whose
  coefficient at the anchor point θ0 is below ε (**note:** this makes the program specialized
  around θ0 and is meant for evaluation, not for training far away from θ0).

`compile()` returns the program object, so several compiles of the same circuit coexist —
just remember that `qc.expvals` always uses the **latest** compile. The retained-term count
is the cost proxy:

In [16]:
def nnz(prog):
    return sum(int(s.mat_const._nnz()) + int(s.mat_cos._nnz()) + int(s.mat_sin._nnz())
               for s in prog.psum_union.steps)

theta_eval = torch.linspace(0.1, 1.2, n_params, dtype=torch.float64)
full = program.expvals(theta_eval)             # `program` = the exact compile from section 2

print(f"{'max_weight':>10} | {'nnz':>6} | max |error| vs exact")
for w in [1, 2, 3]:
    p_w = qc.compile(observables=observables, preset=nb_preset, dtype="float64",
                     max_weight=w, verbosity=0)
    err = float((p_w.expvals(theta_eval) - full).abs().max())
    print(f"{w:>10} | {nnz(p_w):>6} | {err:.3e}")
print(f"{'exact':>10} | {nnz(program):>6} | 0")

max_weight |    nnz | max |error| vs exact
         1 |      1 | 1.110e+00
         2 |     50 | 4.035e-01
         3 |     73 | 3.741e-02
     exact |     85 | 0


## 6) Data embeddings (quantum ML)

A rotation with `embedding_idx=j` takes its angle from a separate **data vector** passed at
evaluation time — a data-encoding layer of a QML model. Embedding inputs are
data, not optimizer parameters (the autograd graph through them stays intact, so a classical layer that produces them still receives its gradient; see `08_hybrid_classical_quantum.ipynb`), and they batch **independently** of θ:
`qc.expvals(theta, embedding=x)` with θ of shape `(p_bs, P)` and x of shape `(e_bs, E)` returns
`(e_bs, p_bs, n_obs)` — every data point × every parameter set in one call.

In [17]:
qml_qc = Circuit(n)
for q in range(n):                    # data-encoding layer: angle = x[q]
    qml_qc.rx(q, embedding_idx=q)
pidx = 0
for q in range(n - 1):                # trainable entangler
    qml_qc.rzz(q, q + 1, param_idx=pidx); pidx += 1
for q in range(n):                    # trainable rotation layer
    qml_qc.ry(q, param_idx=pidx); pidx += 1

qml_qc.compile(observables=[("Z", [0])], preset=nb_preset, dtype="float64")

theta_b = torch.rand(3, qml_qc.n_params, dtype=torch.float64)   # 3 parameter sets
x_b = torch.rand(5, qml_qc.n_embedding, dtype=torch.float64)    # 5 data points
out = qml_qc.expvals(theta_b, embedding=x_b)
print("theta (3, P) x data (5, E) -> output", tuple(out.shape))   # (5, 3, 1)

propagate:   0%|          | 0/17 [00:00<?, ?it/s]

[PPS Info] Propagation complete. Terms generated: 7


zero-filter:   0%|          | 0/17 [00:00<?, ?it/s]

[PPS Info] Terms retained after pruning: 2 (28.571429% of peak)
theta (3, P) x data (5, E) -> output (5, 3, 1)


## 7) Noise channels

Pauli noise propagates through the surrogate exactly: insert `qc.depolarizing(...)` /
`qc.amplitude_damping(...)` into the build like any other gate method.

In [18]:
noisy = Circuit(n)
pidx = 0
for q in range(n):
    noisy.rx(q, param_idx=pidx); pidx += 1
for q in range(n - 1):
    noisy.rzz(q, q + 1, param_idx=pidx); pidx += 1
    noisy.depolarizing(q, 5e-3, 5e-3, 5e-3)       # noise after each entangler
noisy.cnot(0, 1)
for q in range(n):
    noisy.rx(q, param_idx=pidx); pidx += 1

noisy.compile(observables=observables, preset=nb_preset, dtype="float64", verbosity=0)
print("ideal :", [f"{v:+.6f}" for v in program.expvals(theta_eval).tolist()])
print("noisy :", [f"{v:+.6f}" for v in noisy.expvals(theta_eval).tolist()])

ideal : ['+0.645242', '+0.028977', '+1.109788']
noisy : ['+0.632470', '+0.028403', '+1.069895']


## 8) Cross-checking against PennyLane

For small systems (≤ 20 qubits) every compiled program can be validated against a PennyLane
statevector simulation of the same circuit:

In [19]:
ref = program.expvals_reference(theta_eval, max_qubits=20)
dev_max = float((program.expvals(theta_eval) - ref).abs().max())
print("PennyLane reference :", [f"{v:+.6f}" for v in ref.tolist()])
print("max |deviation|     :", f"{dev_max:.3e}")
assert dev_max < 1e-9

PennyLane reference : ['+0.645242', '+0.028977', '+1.109788']
max |deviation|     : 6.217e-15


## Where to go next

- **The rest of `tutorial/`**: the reference API (02), explicit training loops (03),
  batched embedding inputs (04), preset tuning under a GPU budget (05),
  quasi-probability reconstruction (06), and parameter-batch evaluation (07).
- **`../examples/`**: the application cases end to end --- the 127-qubit
  kicked Ising (01), VQE for H2 (02), MaxCut-QAOA (03), and the
  SAFE ma-QAOA surrogate-warmup workflow (04).
